In [0]:
from pyspark.sql.functions import *

In [0]:
bookings_df = spark.table("default.bookings")
members_df = spark.table("default.members")
facilities_df = spark.table("default.facilities")

## Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

In [0]:
result_df = (bookings_df.alias("b").join(
        members_df.alias("m"),
        col("b.memid") == col("m.memid"),
        "inner")
    .filter((col("m.firstname") == "David") &(col("m.surname") == "Farrell"))
    .select(col("b.starttime")))

display(result_df.limit(5))

starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z


How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

In [0]:
result_df = (bookings_df.alias("b").join(
        facilities_df.alias("f"),
        col("b.facid") == col("f.facid"),
        "inner")
    .filter((col("f.name") .like("Tennis Court%")))
    .filter((col("b.starttime") >= "2012-09-21") & (col("b.starttime") <= "2012-09-22"))
    .select(col("b.starttime"),col("f.name"))
    .orderBy(col("b.starttime")))

display(result_df.limit(5))

starttime,name
2012-09-21T08:00:00.000Z,Tennis Court 2
2012-09-21T08:00:00.000Z,Tennis Court 1
2012-09-21T09:30:00.000Z,Tennis Court 1
2012-09-21T10:00:00.000Z,Tennis Court 2
2012-09-21T11:30:00.000Z,Tennis Court 2


How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).


In [0]:
result_df = (members_df.alias("m1").join(
        members_df.alias("m2"),
        col("m1.recommendedby") == col("m2.memid"),
        "left")
    .select(col("m1.firstname").alias("memfname"),col("m1.surname").alias("memsname"),col("m2.firstname").alias("recfname"),col("m2.surname").alias("recsname"))
    .orderBy(col("m1.surname"), col("m1.firstname")))

display(result_df.limit(5))

memfname,memsname,recfname,recsname
Florence,Bader,Ponder,Stibbons
Anne,Baker,Ponder,Stibbons
Timothy,Baker,Jemima,Farrell
Tim,Boothe,Tim,Rownam
Gerald,Butters,Darren,Smith


How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

In [0]:
result_df = (
    bookings_df.alias("b")
    .join(facilities_df.alias("f"), col("b.facid") == col("f.facid"), "inner")
    .join(members_df.alias("m"), col("b.memid") == col("m.memid"), "inner")
    .filter(col("f.name").like("Tennis Court%"))
    .select(concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member"),col("f.name").alias("facility"))
    .distinct()
    .orderBy("member", "facility")
)

display(result_df.limit(5))

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1


How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

In [0]:
name_lookup = {
    row["memid"]: f"{row['firstname']} {row['surname']}"
    for row in members_df.select("memid", "firstname", "surname").collect()
    }

get_recommender_name = udf(lambda x: name_lookup.get(x), StringType())

result_df = (members_df
    .withColumn("member", concat_ws(" ", col("firstname"), col("surname")))
    .withColumn("recommender", get_recommender_name(col("recommendedby")))
    .select("member", "recommender")
    .distinct()
    .orderBy("member"))

display(result_df.limit(5))

member,recommender
Anna Mackenzie,Darren Smith
Anne Baker,Ponder Stibbons
Burton Tracy,null
Charles Owen,Darren Smith
Darren Smith,null


Produce a count of the number of recommendations each member has made. Order by member ID.

In [0]:
result_df = (members_df
    .filter(col("recommendedby").isNotNull())
    .groupBy("recommendedby")
    .agg(count("*").alias("recommendation_count"))
    .select(col("recommendedby").alias("memid"),col("recommendation_count"))
    .orderBy("memid"))

display(result_df.limit(5))

memid,recommendation_count
1,5
2,3
3,1
4,2
5,1


Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.


In [0]:
result_df = (bookings_df
    .groupBy("facid")
    .agg(sum(col("slots")).alias("Total Slots"))
    .select(col("facid"),col("Total Slots"))
    .orderBy("facid"))

display(result_df.limit(5))

facid,Total Slots
0,1320
1,1278
2,1209
3,830
4,1404


Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

In [0]:
result_df = (bookings_df
    .filter((col("starttime") >= "2012-09-01") & (col("starttime") <= "2012-10-01"))
    .groupBy("facid")
    .agg(sum(col("slots")).alias("Total Slots"))
    .select(col("facid"),col("Total Slots"))
    .orderBy("Total Slots"))

display(result_df.limit(5))

facid,Total Slots
5,122
3,422
7,426
8,471
6,540


Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.


In [0]:
result_df = (bookings_df
    .filter(year(col("starttime")) == 2012)
    .groupBy(col("facid"),month(col("starttime")).alias("month"))
    .agg(sum("slots").alias("Total Slots"))
    .orderBy("facid", "month")
)

display(result_df.limit(5))

facid,month,Total Slots
0,7,270
0,8,459
0,9,591
1,7,207
1,8,483


Find the total number of members (including guests) who have made at least one booking.

In [0]:
result_df = bookings_df.agg(countDistinct("memid").alias("Total Members with Bookings"))

display(result_df)

Total Members with Bookings
30


Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.

In [0]:
first_booking_df = (
    bookings_df
    .filter(col("starttime") > "2012-09-01")
    .groupBy("memid")
    .agg(min("starttime").alias("starttime"))
)

result_df = (
    members_df.alias("m")
    .join(first_booking_df.alias("b"), col("m.memid") == col("b.memid"), "inner")
    .select(col("m.firstname"), col("m.surname"),col("m.memid"),col("b.starttime"))
    .orderBy("memid")
)

display(result_df.limit(5))

firstname,surname,memid,starttime
GUEST,GUEST,0,2012-09-01T08:00:00.000Z
Darren,Smith,1,2012-09-01T09:00:00.000Z
Tracy,Smith,2,2012-09-01T11:30:00.000Z
Tim,Rownam,3,2012-09-01T16:00:00.000Z
Janice,Joplette,4,2012-09-01T15:00:00.000Z


Output the names of all members, formatted as 'Surname, Firstname'


In [0]:
result_df = (members_df
    .select(concat_ws(", ", col("surname"), col("firstname")).alias("member_name")))

display(result_df.limit(5))

member_name
"GUEST, GUEST"
"Smith, Darren"
"Smith, Tracy"
"Rownam, Tim"
"Joplette, Janice"


Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.

In [0]:
result_df = (facilities_df
    .filter(lower(col("name")).startswith("tennis")))

display(result_df.limit(5))

facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,Tennis Court 1,5.0,25.0,10000,200
1,Tennis Court 2,5.0,25.0,8000,200


You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.

In [0]:
result_df = (members_df
    .filter(col("telephone").rlike(r"[\(\)]"))
    .select("memid", "telephone")
    .orderBy("memid"))

display(result_df.limit(5))

memid,telephone
0,(000) 000-0000
3,(844) 693-0723
4,(833) 942-4710
5,(844) 078-4130
6,(822) 354-9973


You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

In [0]:
result_df = (members_df
    .select(substring(col("surname"), 1, 1).alias("letter"))
    .groupBy("letter")
    .agg(count("*").alias("count"))
    .orderBy("letter"))

display(result_df.limit(5))

letter,count
B,5
C,2
D,1
F,2
G,2


Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.

In [0]:
result_df = (spark.range(1)
             .select(explode(sequence(
                to_date(lit("2012-10-01")),
                to_date(lit("2012-10-31")))).alias("date"))
             .select(col("date").cast("timestamp").alias("date")))

display(result_df.limit(5))

date
2012-10-01T00:00:00.000Z
2012-10-02T00:00:00.000Z
2012-10-03T00:00:00.000Z
2012-10-04T00:00:00.000Z
2012-10-05T00:00:00.000Z


Return a count of bookings for each month, sorted by month

In [0]:
result_df = (bookings_df
    .groupBy(month(col("starttime")).alias("month"))
    .agg(count("*").alias("booking_count"))
    .orderBy("month"))

display(result_df.limit(5))

month,booking_count
1,1
7,658
8,1472
9,1913
